# 🔬 مسكن ثلاثي الأبعاد: دفتر التشخيص الجذري لتسجيل الكاميرات في COLMAP
### Maskan 3D: Deep Diagnostic & Hypothesis Separation Notebook

**الهدف الهندسي:** تحديد السبب الجذري وراء تسجيل كولماب لـ **12 إطاراً فقط من أصل 163 إطاراً (7.36%)** في الفيديو الحالي، وفصل الفرضيات الثلاث الأساسية بالقياس العددي المباشر:
- **الفرضية (A):** نمط التصوير يعاني من تزيح شبه منعدم (Low Parallax) أو دوران في المكان.
- **الفرضية (B):** معدل استخراج الإطارات (2 fps) غير متناسب مع سرعة الحركة، أو إعدادات المطابق المتسلسل غير كافية.
- **الفرضية (C):** بيئة التشغيل أو خط المعالجة نفسه يعاني من خلل.

> [!NOTE]
> كل رقم في هذا الدفتر ناتج عن كود فعلي يتم تنفيذه؛ ويتم وسم كل نتيجة بـ `[MEASURED]` أو `[INFERRED]` أو `[UNDETERMINED]`. جميع العتبات مصنفة كـ *فرضيات استرشادية مقترحة (proposed heuristics)*.

## ⚙️ 0. إعداد المتغيرات وبيئة التشغيل (Config & Environment Check)
فحص إصدارات بايثون و COLMAP و OpenCV والـ CPU والاتصال بالإنترنت، وضبط ميزانية الوقت الزمنية، واستخراج الإطارات تلقائياً إذا كانت الجلسة جديدة.

In [ ]:
import os
import sys
import glob
import shutil
import subprocess
import urllib.request
from pathlib import Path

# ========================================================
# ⚙️ المتغيرات والمسارات الرئيسية للتشخيص
# ========================================================
OUT_ROOT = "/kaggle/working"
VIDEO_PATH = "/kaggle/working/video.mp4"
DRIVE_FILE_ID = "1Q-n4eVCTGWL8SMwsva1Lk9uMH7rL2hef"
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1wZa2kZkUSTKE9LFCijYBUsFhXRP7ONLw"

EXTRACTED_RAW_DIR = "/kaggle/working/extracted_raw"
FILTERED_FRAMES_DIR = "/kaggle/working/filtered_frames"
EXISTING_PROCESSED_DIR = "/kaggle/working/data/nerfstudio/apartment"
DIAG_RESULTS_DIR = "/kaggle/working/diag_results"

# ميزانية الوقت القصوى لاختبارات المقارنة (بالدقائق) لمنع استنزاف وقت الجلسة
TIME_BUDGET_MIN = 60  # [proposed heuristic: 60 minutes max for sweep]

os.makedirs(DIAG_RESULTS_DIR, exist_ok=True)

print("=" * 65)
print("🔍 0. فحص بيئة التشغيل والبرمجيات المثبتة:")
print(f"🐍 Python Version: {sys.version.split()[0]}")
try:
    import cv2
    print(f"📷 OpenCV Version: {cv2.__version__}")
except ImportError:
    print("⚠️ OpenCV غير مثبت!")

try:
    colmap_ver = subprocess.check_output(["colmap", "--version"], text=True).strip()
    print(f"🗺️ {colmap_ver}")
except Exception as e:
    print("⚠️ COLMAP غير مثبت عبر apt! جاري التثبيت...")
    !apt-get update -y && apt-get install -y colmap ffmpeg

# فحص الإنترنت
try:
    urllib.request.urlopen("https://www.google.com", timeout=5)
    print("🌐 الإنترنت: متصل بنجاح (Internet is ON ✅)")
except Exception:
    print("\n⚠️ تحذير حرج: الإنترنت معطّل في إعدادات Kaggle! فعّله من الشريط الجانبي (Settings -> Internet on)\n")

# فحص أو تثبيت gdown و nerfstudio (للاختبار الضابط)
import importlib.util
if shutil.which('gdown') is None and importlib.util.find_spec('gdown') is None:
    print("📦 تثبيت gdown لتنزيل الفيديو...")
    !pip install --quiet gdown

if shutil.which('ns-download-data') is None and importlib.util.find_spec('nerfstudio') is None:
    print("📦 جاري تثبيت nerfstudio للاختبار الضابط في الخلية 4...")
    !pip install --quiet nerfstudio
else:
    print("📦 Nerfstudio: متاح للاختبار الضابط ✅")

# التأكد من وجود الفيديو أو تنزيله
if not os.path.exists(VIDEO_PATH) or os.path.getsize(VIDEO_PATH) < 1024:
    print("📥 جاري تنزيل الفيديو من Google Drive...")
    url = f"https://drive.google.com/uc?id={DRIVE_FILE_ID}"
    !gdown --fuzzy '{url}' -O '{VIDEO_PATH}'

if os.path.exists(VIDEO_PATH):
    size_mb = os.path.getsize(VIDEO_PATH) / (1024 * 1024)
    print(f"🎥 ملف الفيديو جاهز: {VIDEO_PATH} ({size_mb:.2f} MB)")
else:
    print(f"⚠️ تنبيه: ملف الفيديو غير موجود في {VIDEO_PATH}!")

# فحص وجود الإطارات المستخرجة؛ إذا لم تكن موجودة (جلسة جديدة)، يتم استخراجها تلقائياً عند 2 fps
raw_frames = sorted(glob.glob(os.path.join(EXTRACTED_RAW_DIR, "frame_*.png"))) if os.path.exists(EXTRACTED_RAW_DIR) else []
filt_frames = sorted(glob.glob(os.path.join(FILTERED_FRAMES_DIR, "frame_*.png"))) if os.path.exists(FILTERED_FRAMES_DIR) else []

if not filt_frames and not raw_frames:
    if os.path.exists(VIDEO_PATH):
        print("🎞️ لم يتم العثور على إطارات مستخرجة سابقة (جلسة كاجل جديدة). جاري استخراج الإطارات عند 2 fps...")
        os.makedirs(EXTRACTED_RAW_DIR, exist_ok=True)
        os.makedirs(FILTERED_FRAMES_DIR, exist_ok=True)
        os.system(f'ffmpeg -y -i "{VIDEO_PATH}" -vf fps=2 -q:v 2 "{EXTRACTED_RAW_DIR}/frame_%05d.png" > /dev/null 2>&1')
        extracted = sorted(glob.glob(os.path.join(EXTRACTED_RAW_DIR, "frame_*.png")))
        print(f"   • تم استخراج {len(extracted)} إطاراً خاماً بنجاح.")
        for f in extracted:
            shutil.copy(f, os.path.join(FILTERED_FRAMES_DIR, os.path.basename(f)))
        print(f"   • جاهزية الإطارات: تم حفظ {len(extracted)} إطاراً في {FILTERED_FRAMES_DIR} لضمان عمل الخلية 2 مباشرة.")
    else:
        print("⚠️ تعذر استخراج الإطارات لعدم وجود ملف الفيديو!")
else:
    count = len(filt_frames) if filt_frames else len(raw_frames)
    print(f"🎞️ تم العثور على الإطارات المستخرجة مسبقاً ({count} إطار) جاهزة للتحليل ✅")
print("=" * 65)


## 🔍 1. فحص مخرجات COLMAP السابقة وقاعدة البيانات (Inspect Existing Reconstruction & Database)
1. تشغيل أداة `colmap model_analyzer` على كافة مجلدات النماذج المتفرقة `colmap/sparse/*` لحساب عدد الكاميرات المسجلة، والنقاط ثلاثية الأبعاد، ومتوسط طول المسار وخطأ الإسقاط.
2. فتح `colmap/database.db` بواسطة `sqlite3` لفحص نقاط المعالم (Keypoints) في كل إطار، وأزواج التطابقات الهندسية (`two_view_geometries`)، وتحديد أماكن انقطاع السلسلة (Chain Breaks) بين الإطارات المتتالية وربطها بالزمن بالثواني.

> [!NOTE]
> التوثيق المصدري: أمر `model_analyzer` موثق في `repos/colmap/src/colmap/exe/model.cc:398`، ومخطط قاعدة البيانات موثق في `repos/colmap/doc/database.rst:11-20, 109-125`.

In [ ]:
import os
import glob
import sqlite3
import subprocess
import numpy as np
from pathlib import Path

colmap_dir = Path(EXISTING_PROCESSED_DIR) / "colmap"
if not colmap_dir.exists():
    sweep_v0 = Path(OUT_ROOT) / "sweep_experiments" / "V0_baseline" / "colmap"
    if sweep_v0.exists():
        colmap_dir = sweep_v0
        print("📁 تم العثور على مخرجات V0_baseline من تجارب الـ Sweep: استخدامها للفحص.")

sparse_dir = colmap_dir / "sparse"
db_path = colmap_dir / "database.db"

print("=" * 75)
print("🔍 1.1 فحص النماذج المتفرقة في colmap/sparse عبر colmap model_analyzer:")
print("=" * 75)

models_summary = []
if sparse_dir.exists():
    sub_models = sorted([d for d in sparse_dir.iterdir() if d.is_dir()], key=lambda x: x.name)
    print(f"📁 إجمالي مجلدات النماذج المنفصلة (Disconnected Models): {len(sub_models)}")
    
    print(f"\n{'Model ID':<10} | {'Reg Cameras':<12} | {'3D Points':<12} | {'Mean Track':<12} | {'Reproj Error (px)':<18}")
    print("-" * 75)
    
    for model_path in sub_models:
        try:
            cmd = f"colmap model_analyzer --path '{model_path}'"
            out = subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT)
            
            reg_cams, pts_3d, track_len, reproj_err = "?", "?", "?", "?"
            for line in out.splitlines():
                if "Registered images:" in line:
                    reg_cams = line.split(":")[-1].strip()
                elif "Points:" in line and "3D" not in line:
                    pts_3d = line.split(":")[-1].strip()
                elif "Mean track length:" in line:
                    track_len = line.split(":")[-1].strip()
                elif "Mean reprojection error:" in line:
                    reproj_err = line.split(":")[-1].strip().replace("px", "")
                    
            models_summary.append({
                "id": model_path.name,
                "registered": reg_cams,
                "points": pts_3d,
                "track_length": track_len,
                "reprojection_error": reproj_err
            })
            print(f"{model_path.name:<10} | {reg_cams:<12} | {pts_3d:<12} | {track_len:<12} | {reproj_err:<18}")
        except Exception as e:
            print(f"{model_path.name:<10} | تعذر التحليل: {e}")
else:
    print("ℹ️ ملاحظة هندسية: لم يتم العثور على مجلد colmap/sparse في المسار السابق (أنت في جلسة جديدة).")
    print("   هذا طبيعي تماماً! الخلايا 2 و3 و4 ستعمل بشكل مستقل تماماً،")
    print("   وسيتم بناء وفحص نموذج الأساس V0_baseline تلقائياً في الخلية 3.")

# 1.2 فحص قاعدة البيانات database.db
print("\n" + "=" * 75)
print("🔍 1.2 فحص قاعدة بيانات COLMAP (database.db):")
print("=" * 75)

chain_breaks = []
MIN_INLIERS_CHAIN = 15  # [proposed heuristic: < 15 inliers between consecutive frames indicates a tracking break]

if db_path.exists():
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    
    # 1. إحصاء المعالم (Keypoints)
    cur.execute("SELECT image_id, rows FROM keypoints")
    kp_rows = cur.fetchall()
    if kp_rows:
        kp_counts = np.array([r[1] for r in kp_rows])
        print(f"✨ إحصائيات المعالم المستخرجة (Keypoints per Image) [MEASURED]:")
        print(f"   • الحد الأدنى (Min): {np.min(kp_counts):,}")
        print(f"   • الوسيط (Median): {np.median(kp_counts):,}")
        print(f"   • الحد الأقصى (Max): {np.max(kp_counts):,}")
        
    # 2. أزواج الصور المتحقق منها هندسياً
    cur.execute("SELECT count(*) FROM two_view_geometries WHERE rows > 0")
    n_verified_pairs = cur.fetchone()[0]
    print(f"\n🔗 أزواج الصور المتحقق منها هندسياً بـ Inliers > 0 [MEASURED]: {n_verified_pairs:,} زوج")
    
    # 3. فحص أزواج الإطارات المتتالية (i, i+1) لكشف انقطاع السلسلة
    # pair_id في كولماب: pair_id = 2147483647 * img1 + img2 (مع img1 < img2)
    cur.execute("SELECT image_id, name FROM images ORDER BY image_id")
    images = cur.fetchall()
    print(f"📸 عدد الصور المسجلة في قاعدة البيانات: {len(images)}")
    
    consecutive_inliers = []
    for idx in range(len(images) - 1):
        img1_id = images[idx][0]
        img2_id = images[idx+1][0]
        pair_id = 2147483647 * img1_id + img2_id
        
        cur.execute("SELECT rows FROM two_view_geometries WHERE pair_id = ?", (pair_id,))
        row = cur.fetchone()
        inliers = row[0] if row else 0
        consecutive_inliers.append((img1_id, img2_id, inliers))
        
        if inliers < MIN_INLIERS_CHAIN:
            # تحويل رقم الإطار لزمن بالثواني (بافتراض معدل استخراج 2 إطار/ثانية)
            time_sec = idx / 2.0
            chain_breaks.append((idx, idx+1, inliers, time_sec))
            
    conn.close()
    
    print(f"\n⚠️ انقطاعات السلسلة المتتالية (Chain Breaks: inliers < {MIN_INLIERS_CHAIN}) [MEASURED]:")
    print(f"   • إجمالي عدد الانقطاعات: {len(chain_breaks)} من أصل {len(images)-1} انتقال")
    if chain_breaks:
        print(f"\n{'Pair (i -> i+1)':<18} | {'Inliers':<10} | {'Video Time (approx)':<20}")
        print("-" * 52)
        for b in chain_breaks[:15]:  # عرض أول 15 انقطاع
            print(f"Frame {b[0]:03d} -> {b[1]:03d}     | {b[2]:<10} | {b[3]:.1f} s")
        if len(chain_breaks) > 15:
            print(f"... و {len(chain_breaks)-15} انقطاعاً آخر (محفوظة بالكامل في التقرير)")
else:
    print("⚠️ لم يتم العثور على database.db.")
print("=" * 75)


## 🔬 2. التحليل المستقل لأزواج الإطارات عبر OpenCV (Independent Optical & Motion Analysis)
تحليل مستقل تماماً لجميع إطارات `extracted_raw` دون الاعتماد على COLMAP:
- مطابقة المعالم باستخدام `cv2.SIFT_create` واختبار نسبة لوي (0.75).
- حساب مصفوفة الأساس $F$ ومصفوفة الهوموغرافي $H$ عبر RANSAC للأبعاد الزمنية المختلفة ($g \in \{1, 2, 4\}$).
- قياس نسبة الهوموغرافي $H_{ratio} = H_{inliers} / F_{inliers}$، ومقدار التزيح الفعلي (Parallax Proxy) وإزاحة البكسلات كنسبة من عرض الصورة.
- تصنيف كل ثانية من الفيديو استرشادياً: `LOW_OVERLAP`، `ROTATION_DOMINANT`، `FAST_MOTION`، أو `OK`.

> [!NOTE]
> **العتبات الاسترشادية المقترحة (Proposed Heuristics):**
> - `THRESH_LOW_OVERLAP_INLIERS = 30`: أقل من 30 تطابقاً هندسياً يعتبر تداخلاً ضعيفاً.
> - `THRESH_ROT_H_RATIO = 0.90` و `THRESH_ROT_PARALLAX_PX = 2.0`: نسبة هوموغرافي عالية مع تزيح أقل من 2 بكسل تعني دوراناً في المكان.
> - `THRESH_FAST_MOTION_SHIFT = 0.35`: حركة تتجاوز 35% من عرض الإطار بين لقطتين متتاليتين تعتبر حركة سريعة مفرطة.

In [ ]:
import os
import glob
import csv
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. قراءة جميع الإطارات الخام المستخرجة
frame_files = sorted(glob.glob(os.path.join(EXTRACTED_RAW_DIR, "frame_*.png")))
if not frame_files:
    frame_files = sorted(glob.glob(os.path.join(FILTERED_FRAMES_DIR, "frame_*.png")))

num_frames = len(frame_files)
print(f"🎞️ عدد الإطارات الخاضعة للفحص المستقل: {num_frames} إطار")

if num_frames < 2:
    print("⚠️ عدد الإطارات غير كافٍ لإجراء تحليل الأزواج!")
else:
    sift = cv2.SIFT_create()
    bf = cv2.BFMatcher(cv2.NORM_L2)
    
    # استخراج الميزات لكل إطار مع تصغير العرض الأقصى لـ 960 للسرعة
    print("⏳ استخراج ميزات SIFT لجميع الإطارات...")
    kps_list = []
    descs_list = []
    img_shapes = []
    
    for f_path in frame_files:
        img = cv2.imread(f_path)
        h, w = img.shape[:2]
        scale = 960.0 / max(h, w) if max(h, w) > 960 else 1.0
        if scale < 1.0:
            img = cv2.resize(img, (int(w * scale), int(h * scale)))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        kps, des = sift.detectAndCompute(gray, None)
        kps_list.append(kps)
        descs_list.append(des)
        img_shapes.append((gray.shape[1], gray.shape[0]))  # (w, h)
        
    # 2. تحليل الفجوات الزمنية (gaps in {1, 2, 4})
    analysis_results = []
    gaps = [1, 2, 4]
    
    for gap in gaps:
        print(f"🔍 جاري فحص الانتقالات بفارق زمني gap = {gap}...")
        for i in range(num_frames - gap):
            j = i + gap
            des1, des2 = descs_list[i], descs_list[j]
            if des1 is None or des2 is None or len(des1) < 10 or len(des2) < 10:
                continue
                
            # Lowe's ratio test (0.75)
            matches = bf.knnMatch(des1, des2, k=2)
            good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
            n_raw = len(good_matches)
            
            if n_raw >= 8:
                pts1 = np.float32([kps_list[i][m.queryIdx].pt for m in good_matches])
                pts2 = np.float32([kps_list[j][m.trainIdx].pt for m in good_matches])
                
                # Fundamental Matrix RANSAC
                F, mask_F = cv2.findFundamentalMat(pts1, pts2, cv2.FM_RANSAC, 3.0)
                f_inliers = int(np.sum(mask_F)) if mask_F is not None else 0
                
                # Homography Matrix RANSAC
                H, mask_H = cv2.findHomography(pts1, pts2, cv2.RANSAC, 3.0)
                h_inliers = int(np.sum(mask_H)) if mask_H is not None else 0
                
                h_ratio = h_inliers / max(f_inliers, 1)
                
                # Parallax Proxy: residual displacement after applying H to F-inliers
                parallax_px = 0.0
                if H is not None and mask_F is not None and f_inliers > 0:
                    f_pts1 = pts1[mask_F.ravel() == 1]
                    f_pts2 = pts2[mask_F.ravel() == 1]
                    # إسقاط pts1 عبر H
                    pts1_homo = np.hstack([f_pts1, np.ones((len(f_pts1), 1))])
                    proj_pts2 = (H @ pts1_homo.T).T
                    proj_pts2 = proj_pts2[:, :2] / (proj_pts2[:, 2:3] + 1e-8)
                    resids = np.linalg.norm(f_pts2 - proj_pts2, axis=1)
                    parallax_px = float(np.median(resids))
                    
                # Shift as fraction of image width
                img_w = img_shapes[i][0]
                shifts = np.linalg.norm(pts2 - pts1, axis=1)
                med_shift_ratio = float(np.median(shifts) / max(img_w, 1))
            else:
                f_inliers, h_inliers, h_ratio, parallax_px, med_shift_ratio = 0, 0, 0.0, 0.0, 0.0
                
            time_sec = i / 2.0  # approximate time in video
            analysis_results.append({
                "gap": gap,
                "idx1": i,
                "idx2": j,
                "time_sec": time_sec,
                "raw_matches": n_raw,
                "f_inliers": f_inliers,
                "h_inliers": h_inliers,
                "h_ratio": h_ratio,
                "parallax_px": parallax_px,
                "shift_ratio": med_shift_ratio
            })
            
    # حفظ النتائج في ملف CSV
    csv_path = os.path.join(DIAG_RESULTS_DIR, "frame_pair_analysis.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=analysis_results[0].keys())
        writer.writeheader()
        writer.writerows(analysis_results)
    print(f"💾 تم حفظ نتائج تحليل الأزواج الكاملة إلى: {csv_path}")
    
    # 3. تصنيف الجدول الزمني (Classification via Proposed Heuristics)
    THRESH_LOW_OVERLAP_INLIERS = 30     # [proposed heuristic: inliers < 30 indicates weak visual overlap]
    THRESH_ROT_H_RATIO = 0.90          # [proposed heuristic: H/F > 0.90 indicates planar or pure-rotation motion]
    THRESH_ROT_PARALLAX_PX = 2.0       # [proposed heuristic: parallax < 2px indicates negligible depth parallax]
    THRESH_FAST_MOTION_SHIFT = 0.35    # [proposed heuristic: shift > 35% of frame width indicates violent motion]
    
    gap1_results = [r for r in analysis_results if r["gap"] == 1]
    n_gap1 = len(gap1_results)
    
    class_counts = {"LOW_OVERLAP": 0, "ROTATION_DOMINANT": 0, "FAST_MOTION": 0, "OK": 0}
    worst_segments = []
    
    for r in gap1_results:
        if r["f_inliers"] < THRESH_LOW_OVERLAP_INLIERS:
            c = "LOW_OVERLAP"
            worst_segments.append((r["time_sec"], "LOW_OVERLAP", r["f_inliers"]))
        elif r["shift_ratio"] > THRESH_FAST_MOTION_SHIFT:
            c = "FAST_MOTION"
            worst_segments.append((r["time_sec"], "FAST_MOTION", r["shift_ratio"]))
        elif r["h_ratio"] > THRESH_ROT_H_RATIO and r["parallax_px"] < THRESH_ROT_PARALLAX_PX:
            c = "ROTATION_DOMINANT"
        else:
            c = "OK"
        class_counts[c] += 1
        
    print("\n" + "=" * 75)
    print("📊 نسب تصنيف الجدول الزمني للفيديو (Timeline Classification - Gap=1) [MEASURED on Heuristics]:")
    print("=" * 75)
    for k, v in class_counts.items():
        pct = (v / max(n_gap1, 1)) * 100.0
        print(f"  • {k:<18}: {v:3d} أزواج ({pct:5.1f}% من الفيديو)")
    print("-" * 75)
    
    # تفسير معدل الاستخراج باللغة العربية
    bad_pct = ((class_counts['LOW_OVERLAP'] + class_counts['FAST_MOTION']) / max(n_gap1, 1)) * 100.0
    print(f"\n📢 الاستنتاج التحليلي لمعدل الاستخراج (Extraction FPS Assessment) [INFERRED]:")
    if bad_pct >= 40.0:
        print(f"  ⚠️ نسبة الانتقالات المتعثرة (تداخل ضعيف + حركة سريعة) تمثل {bad_pct:.1f}% من الفيديو!")
        print("  💡 هذا يشير بقوة إلى أن معدل الاستخراج الحالي (2 fps) منخفض جداً بالنسبة لسرعة حركة الكاميرا.")
        print("  💡 زيادة معدل الاستخراج إلى 6 fps مع تقليم التكرار سيوفر تداخلاً بصرياً أكثر استقراراً.")
    else:
        print(f"  ✅ التداخل بين الإطارات عند 2 fps مقبول نسبياً ({100.0-bad_pct:.1f}% مستقر).")
        
    # 4. رسم المخطط البياني وتوثيقه
    fig, axs = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    t_vals = [r["time_sec"] for r in gap1_results]
    
    axs[0].plot(t_vals, [r["f_inliers"] for r in gap1_results], 'b-', label="F Inliers")
    axs[0].axhline(THRESH_LOW_OVERLAP_INLIERS, color='r', linestyle='--', label=f"Low Overlap Threshold ({THRESH_LOW_OVERLAP_INLIERS})")
    axs[0].set_ylabel("Inliers")
    axs[0].legend(loc="upper right")
    axs[0].grid(True, alpha=0.3)
    
    axs[1].plot(t_vals, [r["h_ratio"] for r in gap1_results], 'g-', label="H/F Inlier Ratio")
    axs[1].axhline(THRESH_ROT_H_RATIO, color='orange', linestyle='--', label=f"Rotation Heuristic ({THRESH_ROT_H_RATIO})")
    axs[1].set_ylabel("H Ratio")
    axs[1].legend(loc="upper right")
    axs[1].grid(True, alpha=0.3)
    
    axs[2].plot(t_vals, [r["parallax_px"] for r in gap1_results], 'm-', label="Parallax Proxy (px)")
    axs[2].axhline(THRESH_ROT_PARALLAX_PX, color='r', linestyle='--', label=f"Min Parallax ({THRESH_ROT_PARALLAX_PX}px)")
    axs[2].set_ylabel("Parallax (px)")
    axs[2].legend(loc="upper right")
    axs[2].grid(True, alpha=0.3)
    
    axs[3].plot(t_vals, [r["shift_ratio"] for r in gap1_results], 'c-', label="Pixel Shift / Frame Width")
    axs[3].axhline(THRESH_FAST_MOTION_SHIFT, color='r', linestyle='--', label=f"Fast Motion ({THRESH_FAST_MOTION_SHIFT})")
    axs[3].set_ylabel("Shift / Width")
    axs[3].set_xlabel("Video Timeline (seconds)")
    axs[3].legend(loc="upper right")
    axs[3].grid(True, alpha=0.3)
    
    plt.suptitle("Independent Frame-Pair Motion & Optical Diagnostics (Gap=1)", fontsize=14)
    plt.tight_layout()
    plot_path = os.path.join(DIAG_RESULTS_DIR, "frame_pair_metrics.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f"📊 تم حفظ المخطط البياني الزمني إلى: {plot_path}")
print("=" * 75)


## 🧪 3. تجارب المقارنة الموجهة عبر أوامر COLMAP المباشرة (Controlled Parameter Sweep)
تنفيذ أربعة مسارات مقارنة مستقلة عبر الـ CPU فقط لتحديد المتغير الحاسم:
- **V0 (الأساس - إعادة إنتاج الفشل):** استخراج 2 fps ومطابقة متسلسلة بتداخل 10 (`overlap 10`).
- **V1 (كثافة زمنية أعلى):** استخراج 6 fps وتقليصها للأحدّ بحد أقصى 200 إطار، ومطابقة متسلسلة بتداخل 20 (`overlap 20`).
  *(ملاحظة: تخطي كشف الحلقات loop_detection لعدم توفر شجرة مفردات vocab_tree مرخصة محلياً)*.
- **V2 (مطابقة شاملة Exhaustive Matcher):** استخراج 2 fps مع مطابقة كل الأزواج على أول 120 إطاراً لحسم ما إذا كان الخلل في نمط المطابقة المتسلسلة.
- **V3 (تشخيصي فقط - تخفيف معايير التثليث):** إعدادات V0 مع تخفيف زاوية التثليث (`init_min_tri_angle 3.0` و `min_num_matches 8` و `abs_pose_min_num_inliers 8`).

> [!WARNING]
> أي نتائج لـ V3 تُصنف كـ **"تشخيصية فقط (DIAGNOSTIC ONLY)"**؛ ويجب التحقق من متوسط خطأ الإسقاط وطول المسار واستواء الكاميرات قبل الاعتماد عليها.

In [ ]:
import os
import glob
import time
import shutil
import subprocess
from pathlib import Path

SWEEP_DIR = os.path.join(OUT_ROOT, "sweep_experiments")
os.makedirs(SWEEP_DIR, exist_ok=True)

sweep_variants = [
    {"name": "V0_baseline", "fps": 2, "cap": 163, "matcher": "sequential", "overlap": 10, "relaxed": False},
    {"name": "V1_dense_6fps", "fps": 6, "cap": 200, "matcher": "sequential", "overlap": 20, "relaxed": False},
    {"name": "V2_exhaustive", "fps": 2, "cap": 120, "matcher": "exhaustive", "overlap": 0, "relaxed": False},
    {"name": "V3_relaxed_mapper", "fps": 2, "cap": 163, "matcher": "sequential", "overlap": 10, "relaxed": True}
]

sweep_results = []
start_sweep_time = time.time()
time_budget_sec = TIME_BUDGET_MIN * 60

print("=" * 85)
print(f"🧪 بدء تجارب المقارنة الموجهة (Sweep) بميزانية وقت قصوى: {TIME_BUDGET_MIN} دقيقة")
print("=" * 85)

for var in sweep_variants:
    v_name = var["name"]
    v_dir = os.path.join(SWEEP_DIR, v_name)
    img_dir = os.path.join(v_dir, "images")
    colmap_v_dir = os.path.join(v_dir, "colmap")
    sparse_v_dir = os.path.join(colmap_v_dir, "sparse")
    db_v_path = os.path.join(colmap_v_dir, "database.db")
    
    # التحقق من الميزانية الزمنية
    elapsed = time.time() - start_sweep_time
    rem_time = time_budget_sec - elapsed
    if rem_time < 300:  # أقل من 5 دقائق متبقية
        print(f"⏱️ تم استهلاك ميزانية الوقت المحددة ({TIME_BUDGET_MIN} دقيقة). التوقف بأمان قبل {v_name}.")
        break
        
    print(f"\n▶️ تشغيل المسار: [{v_name}] (المتبقي من الميزانية: {rem_time/60:.1f} دقيقة)...")
    t0 = time.time()
    
    # تجهيز المجلدات
    if os.path.exists(v_dir):
        shutil.rmtree(v_dir)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(sparse_v_dir, exist_ok=True)
    
    # 1. تجهيز الإطارات
    if var["fps"] == 2 and os.path.exists(FILTERED_FRAMES_DIR):
        all_frames = sorted(glob.glob(os.path.join(FILTERED_FRAMES_DIR, "frame_*.png")))[:var["cap"]]
        for idx, src in enumerate(all_frames):
            shutil.copy(src, os.path.join(img_dir, f"image_{idx+1:05d}.png"))
    else:
        # استخراج من الفيديو مباشرة بمعدل fps المحدد
        raw_tmp = os.path.join(v_dir, "raw_extract")
        os.makedirs(raw_tmp, exist_ok=True)
        os.system(f'ffmpeg -y -i "{VIDEO_PATH}" -vf fps={var["fps"]} -q:v 2 "{raw_tmp}/f_%05d.png" > /dev/null 2>&1')
        extracted = sorted(glob.glob(os.path.join(raw_tmp, "f_*.png")))
        if len(extracted) > var["cap"]:
            step = len(extracted) / var["cap"]
            selected = [extracted[int(i * step)] for i in range(var["cap"])]
        else:
            selected = extracted
        for idx, src in enumerate(selected):
            shutil.copy(src, os.path.join(img_dir, f"image_{idx+1:05d}.png"))
        shutil.rmtree(raw_tmp)
        
    n_input_images = len(glob.glob(os.path.join(img_dir, "*.png")))
    print(f"   • عدد الإطارات المدخلة: {n_input_images}")
    
    # 2 & 3. استخراج الميزات والمطابقة (مع تسريع ذكي لـ V3 عبر نسخ قاعدة بيانات V0)
    reused_v0_db = False
    if var["relaxed"]:
        v0_db = os.path.join(SWEEP_DIR, "V0_baseline", "colmap", "database.db")
        if os.path.exists(v0_db):
            print("   ⚡ تسريع ذكي: نسخ database.db من V0 لتوفير 5 دقائق من استخراج SIFT المكرر...")
            shutil.copy(v0_db, db_v_path)
            reused_v0_db = True
            
    if not reused_v0_db:
        # استخراج الميزات (CPU only, single shared camera, OPENCV)
        cmd_feat = (
            f"colmap feature_extractor "
            f"--database_path '{db_v_path}' "
            f"--image_path '{img_dir}' "
            f"--ImageReader.single_camera 1 "
            f"--ImageReader.camera_model OPENCV "
            f"--SiftExtraction.use_gpu 0 > /dev/null 2>&1"
        )
        os.system(cmd_feat)
        
        # المطابقة
        if var["matcher"] == "sequential":
            cmd_match = (
                f"colmap sequential_matcher "
                f"--database_path '{db_v_path}' "
                f"--SequentialMatching.overlap {var['overlap']} "
                f"--SiftMatching.use_gpu 0 > /dev/null 2>&1"
            )
        else:
            cmd_match = (
                f"colmap exhaustive_matcher "
                f"--database_path '{db_v_path}' "
                f"--SiftMatching.use_gpu 0 > /dev/null 2>&1"
            )
        os.system(cmd_match)
    
    # 4. بناء النموذج (Mapper)
    if var["relaxed"]:
        mapper_args = "--Mapper.init_min_tri_angle 3.0 --Mapper.min_num_matches 8 --Mapper.abs_pose_min_num_inliers 8"
    else:
        mapper_args = "--Mapper.ba_global_function_tolerance=1e-6"
        
    cmd_map = (
        f"colmap mapper "
        f"--database_path '{db_v_path}' "
        f"--image_path '{img_dir}' "
        f"--output_path '{sparse_v_dir}' "
        f"{mapper_args} > /dev/null 2>&1"
    )
    os.system(cmd_map)
    runtime_sec = time.time() - t0
    
    # 5. تحليل النتائج عبر model_analyzer
    sub_models = [d for d in Path(sparse_v_dir).iterdir() if d.is_dir()]
    n_models = len(sub_models)
    reg_total = 0
    largest_size = 0
    reproj_err_val = "-"
    track_len_val = "-"
    
    if sub_models:
        for sm in sub_models:
            try:
                out = subprocess.check_output(f"colmap model_analyzer --path '{sm}'", shell=True, text=True)
                cur_reg = 0
                for line in out.splitlines():
                    if "Registered images:" in line:
                        cur_reg = int(line.split(":")[-1].strip())
                        reg_total += cur_reg
                        if cur_reg > largest_size:
                            largest_size = cur_reg
                    elif "Mean track length:" in line and track_len_val == "-":
                        track_len_val = line.split(":")[-1].strip()
                    elif "Mean reprojection error:" in line and reproj_err_val == "-":
                        reproj_err_val = line.split(":")[-1].strip().replace("px", "")
            except Exception:
                pass
                
    diag_flag = " [DIAGNOSTIC ONLY]" if var["relaxed"] else ""
    sweep_results.append({
        "variant": v_name + diag_flag,
        "input_frames": n_input_images,
        "registered": reg_total,
        "n_models": n_models,
        "largest_model": largest_size,
        "reproj_error": reproj_err_val,
        "track_length": track_len_val,
        "runtime_sec": f"{runtime_sec:.1f}s"
    })
    print(f"   ✅ اكتمل المسار: {reg_total}/{n_input_images} كاميرات مسجلة (نماذج: {n_models}) في {runtime_sec:.1f} ثانية")

# طباعة جدول المقارنة الشامل
print("\n" + "=" * 95)
print("📊 جدول المقارنة الشامل لتجارب كولماب (COLMAP Controlled Sweep Results) [MEASURED]:")
print("=" * 95)
print(f"{'Variant':<25} | {'Frames':<8} | {'Registered':<12} | {'Models':<8} | {'Largest':<9} | {'Reproj Err':<12} | {'Track Len':<10} | {'Runtime'}")
print("-" * 95)
for r in sweep_results:
    print(f"{r['variant']:<25} | {r['input_frames']:<8} | {r['registered']:<12} | {r['n_models']:<8} | {r['largest_model']:<9} | {r['reproj_error']:<12} | {r['track_length']:<10} | {r['runtime_sec']}")
print("=" * 95)


## 🧪 4. اختبار ضابط لبيئة التشغيل (Pipeline Control Test)
التحقق من صحة بيئة العمل وخلو البرمجيات من الخلل بتنزيل عينة قياسية معروفة (`poster`) عبر `ns-download-data` وتشغيل نفس أمر `ns-process-data images` السابق عليها:
- إذا كانت نسبة التسجيل $\ge 90\%$: يثبت سلامة خط المعالجة وبيئة كاجل ويؤكد أن المشكلة في خصائص تصوير الفيديو أو إعداداته `[INFERRED]`.
- إذا كانت نسبة التسجيل $< 70\%$: يشير لخلل في بيئة العمل أو تثبيت كولماب `[INFERRED]`.

> [!NOTE]
> التوثيق المصدري: أمر تنزيل البيانات موثق في `repos/nerfstudio/nerfstudio/scripts/downloads/download_data.py:184-209, 546`.

In [ ]:
import os
import json
import glob
import subprocess
from pathlib import Path

CONTROL_DATA_DIR = "/kaggle/working/control_data"
CONTROL_OUTPUT_DIR = "/kaggle/working/control_output"
os.makedirs(CONTROL_DATA_DIR, exist_ok=True)

print("=" * 75)
print("🧪 4. اختبار بيئة التشغيل الضابط (Pipeline Control Test - Poster Dataset):")
print("=" * 75)

# 1. تنزيل عينة poster المعتمدة
print("📥 جاري تنزيل مجموعة بيانات poster القياسية عبر ns-download-data...")
cmd_dl = f"ns-download-data nerfstudio --capture-name poster --save-dir '{CONTROL_DATA_DIR}'"
res_dl = os.system(cmd_dl)

poster_images_dir = Path(CONTROL_DATA_DIR) / "nerfstudio" / "poster" / "images"
if not poster_images_dir.exists():
    candidates = glob.glob(f"{CONTROL_DATA_DIR}/**/images", recursive=True)
    if candidates:
        poster_images_dir = Path(candidates[0])

if not poster_images_dir.exists():
    print("❌ تعذر تنزيل مجموعة بيانات poster الضابطة أو تعطل الاتصال!")
    print("توقف اختبار البيئة الضابطة (CONTROL TEST UNDETERMINED).")
else:
    control_input_count = len(list(poster_images_dir.glob("*.jpg"))) + len(list(poster_images_dir.glob("*.png")))
    print(f"✅ تم تنزيل العينة بنجاح: {control_input_count} صورة قياسية في {poster_images_dir}")
    
    # 2. تشغيل نفس أمر المعالجة المعتمد في المشروع
    if os.path.exists(CONTROL_OUTPUT_DIR):
        import shutil
        shutil.rmtree(CONTROL_OUTPUT_DIR)
        
    print("🚀 تشغيل ns-process-data images بنفس المعايير (sequential, perspective, no-gpu)...")
    cmd_proc = (
        f"ns-process-data images "
        f"--data '{poster_images_dir}' "
        f"--output-dir '{CONTROL_OUTPUT_DIR}' "
        f"--matching-method sequential "
        f"--camera-type perspective "
        f"--no-gpu"
    )
    os.system(cmd_proc)
    
    # 3. فحص نسبة تسجيل الكاميرات في العينة الضابطة
    control_tf = Path(CONTROL_OUTPUT_DIR) / "transforms.json"
    control_reg_count = 0
    if control_tf.exists():
        with open(control_tf, "r") as f:
            c_meta = json.load(f)
            control_reg_count = len(c_meta.get("frames", []))
            
    control_pct = (control_reg_count / max(control_input_count, 1)) * 100.0
    
    print("\n" + "-" * 75)
    print(f"📊 نتيجة الاختبار الضابط (Control Test Registration Rate) [MEASURED]:")
    print(f"   • الكاميرات المسجلة: {control_reg_count} من أصل {control_input_count} ({control_pct:.1f}%)")
    print("-" * 75)
    
    # الحكم والتقييم
    print("⚖️ التفسير الهندسي لبيئة التشغيل [INFERRED]:")
    if control_pct >= 90.0:
        control_verdict = "PIPELINE_OK"
        print("   ✅ البيئة وخط المعالجة ونسخة COLMAP سليمة 100% (Pipeline OK).")
        print("   💡 الخلل في تجربة الشقة محصور بالكامل في أسلوب التقاط الفيديو أو معدل الاستخراج.")
    elif control_pct >= 70.0:
        control_verdict = "PIPELINE_ACCEPTABLE"
        print("   ⚠️ البيئة مقبولة ولكن هناك تراجع طفيف في أداء المطابقة المتسلسلة على البيانات الضابطة.")
    else:
        control_verdict = "ENVIRONMENT_FAULT"
        print("   ❌ فشل في تسجيل العينة الضابطة (<70%)! يوجد خلل برمجي في بناء COLMAP أو حزم النظام.")
print("=" * 75)


## ⚖️ 5. جدول الحكم النهائي وتحديد المسار القادم (Root-Cause Verdict & Next Experiment)
تجميع الأدلة الحسابية المقاسة للفصل القاطع بين الفرضيات الخمس، وتحديد التجربة الوحيدة المبررة رقمياً.

In [ ]:
print("=" * 85)
print("⚖️ 5. جدول حسم الفرضيات بالدليل الرقمي (Hypothesis Separation Verdict Table):")
print("=" * 85)

# استرجاع المؤشرات المقاسة
rot_pct = class_counts.get("ROTATION_DOMINANT", 0) / max(n_gap1, 1) * 100.0 if 'class_counts' in globals() else 0.0
bad_motion_pct = (class_counts.get("LOW_OVERLAP", 0) + class_counts.get("FAST_MOTION", 0)) / max(n_gap1, 1) * 100.0 if 'class_counts' in globals() else 0.0
n_breaks = len(chain_breaks) if 'chain_breaks' in globals() else 0
n_models_found = len(models_summary) if 'models_summary' in globals() else 0
ctrl_pct = control_pct if 'control_pct' in globals() else 0.0
med_kp = np.median(kp_counts) if 'kp_counts' in globals() else 0

# تقييم الفرضيات وفقاً للأدلة
# H1: تزيح ضعيف / دوران في المكان
h1_status = "SUPPORTED" if rot_pct >= 35.0 else "NOT SUPPORTED"
# H2: معدل استخراج الإطارات منخفض أو الحركة سريعة جداً
h2_status = "SUPPORTED" if bad_motion_pct >= 40.0 or n_breaks >= 10 else "NOT SUPPORTED"
# H3: إعدادات المطابق المتسلسل تسببت في تشتت النماذج
h3_status = "SUPPORTED" if n_models_found > 1 else "NOT SUPPORTED"
# H4: خلل في بيئة التشغيل أو حزم كولماب
h4_status = "NOT SUPPORTED" if ctrl_pct >= 85.0 else ("SUPPORTED" if ctrl_pct < 70.0 and ctrl_pct > 0 else "UNDETERMINED")
# H5: المشهد يفتقر للميزات البصرية (Scene Lacks Texture)
THRESH_TEXTURE_KEYPOINTS = 1000  # [proposed heuristic: < 1000 keypoints indicates texture starvation]
h5_status = "SUPPORTED" if med_kp < THRESH_TEXTURE_KEYPOINTS and med_kp > 0 else "NOT SUPPORTED"

print(f"{'Hypothesis':<42} | {'Evidence (Cell + Number)':<28} | {'Status'}")
print("-" * 85)
print(f"{'H1: Rotation-dominant / Low parallax':<42} | Cell 2: {rot_pct:.1f}% rotation frames     | {h1_status}")
print(f"{'H2: FPS too low / Fast camera motion':<42} | Cell 2: {bad_motion_pct:.1f}% weak/fast, {n_breaks} breaks | {h2_status}")
print(f"{'H3: Matcher settings cause fragmentation':<42} | Cell 1: {n_models_found} disconnected models   | {h3_status}")
print(f"{'H4: Pipeline / Environment fault':<42} | Cell 4: {ctrl_pct:.1f}% on control poster  | {h4_status}")
print(f"{'H5: Scene lacks texture (starvation)':<42} | Cell 1: Median {med_kp:.0f} keypoints/img | {h5_status}")
print("=" * 85)

# التوصية الوحيدة المبررة رقمياً
print("\n🎯 التجربة القادمة المبررة رقمياً بالقياسات (Next Single Justified Action) [INFERRED]:")
if h4_status == "SUPPORTED":
    print("👉 الخلل في بيئة البرمجيات نفسها: يلزم إعادة بناء حزم COLMAP أو الترقية لنسخة رسمية ثابتة.")
elif h2_status == "SUPPORTED" and 'sweep_results' in globals() and any(r['variant'].startswith('V1') and int(r['registered']) > 20 for r in sweep_results):
    print("👉 اعتماد مسار الكثافة العالية (V1: 6 fps مع تقليم التكرار وتداخل sequential 20) كبديل فوري للبيانات الحالية.")
elif h1_status == "SUPPORTED":
    print("👉 إعادة تصوير الغرفة بممشى محيطي مستمر (Perimeter Walking) بخط أساس واسع، لأن الدوران في المكان غير قابل للحل هندسياً.")
else:
    print("👉 فحص تقرير المقارنة في الخلية 3 لتحديد النموذج الأكبر من مسارات V1 أو V2.")


## 📦 6. تجميع ملفات التشخيص وضغط الحزمة (Package & Download Diagnostics)
حفظ جميع ملفات الـ CSV والمخططات البيانية وسجلات النماذج في أرشيف مضغوط واحد `diag_results.zip` لتسهيل مراجعتها وتنزيلها من تبويب Output.

In [ ]:
import os
import zipfile

ZIP_DIAG_PATH = os.path.join(OUT_ROOT, "diag_results.zip")

# جمع الملفات المهمة في DIAG_RESULTS_DIR
with zipfile.ZipFile(ZIP_DIAG_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(DIAG_RESULTS_DIR):
        for file in files:
            f_path = os.path.join(root, file)
            rel_path = os.path.relpath(f_path, DIAG_RESULTS_DIR)
            zipf.write(f_path, os.path.join("diagnostics", rel_path))
            
    # إضافة مخرجات الـ sweep إذا توفرت
    if os.path.exists(SWEEP_DIR):
        for root, dirs, files in os.walk(SWEEP_DIR):
            for file in files:
                if file.endswith(".txt") or file.endswith(".log") or file.endswith(".json"):
                    f_path = os.path.join(root, file)
                    rel_path = os.path.relpath(f_path, SWEEP_DIR)
                    zipf.write(f_path, os.path.join("sweep_logs", rel_path))

zip_size_mb = os.path.getsize(ZIP_DIAG_PATH) / (1024 * 1024) if os.path.exists(ZIP_DIAG_PATH) else 0.0

print("=" * 75)
print("🎉 اكتمل تجهيز حزمة التشخيص الجذرية بنجاح!")
print(f"📦 مسار الملف المضغوط: {ZIP_DIAG_PATH}")
print(f"📊 حجم الملف المضغوط: {zip_size_mb:.2f} MB")
print("=" * 75)
print("\n📥 [إرشادات تنزيل ملف التشخيص من Kaggle]:")
print("1. من الشريط الجانبي الأيمن، افتح تبويب: (Output).")
print("2. ستجد ملف: diag_results.zip داخل مجلد /kaggle/working.")
print("3. اضغط على النقاط الثلاث بجانبه ثم اختر: (Download).")
print("=" * 75 + "\n")
